# Phase 4a: Random Forest Model (Week 10)

Train and evaluate a Random Forest model on combined user-destination features.

**Input:** Phases 1-3 cached data (user/destination profiles + weather + embeddings)

**Output:** Trained model + evaluation metrics (Recall@10, NDCG@10, MRR)

In [10]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from config import *
from src.models.random_forest import RandomForestRecommender
from src.evaluation.harness import RecommendationHarness

print("="*70)
print("PHASE 4a: Random Forest Model")
print("="*70)

PHASE 4a: Random Forest Model


## Step 1: Load Phase 1-3 Cached Data

In [11]:
print("\n[1/6] Loading Phase 1 data...")

# Load training and test sets
train_reviews = pd.read_parquet(CACHE_DIR / 'train_reviews.parquet')
test_reviews = pd.read_parquet(CACHE_DIR / 'test_reviews.parquet')
businesses = pd.read_parquet(CACHE_DIR / 'businesses_major_cities.parquet')

print(f"  ✓ Training reviews: {len(train_reviews):,}")
print(f"  ✓ Test reviews: {len(test_reviews):,}")
print(f"  ✓ Businesses: {len(businesses):,}")
print(f"  ✓ Cities: {businesses['city'].nunique()}")


[1/6] Loading Phase 1 data...
  ✓ Training reviews: 221
  ✓ Test reviews: 77
  ✓ Businesses: 10,341
  ✓ Cities: 39


## Step 2: Load Weather Features (Phase 2)

In [12]:
print("\n[2/6] Loading Phase 2 weather features...")

weather_features = pd.read_parquet(CACHE_DIR / 'weather_features_for_models.parquet')
print(f"  ✓ Weather features: {weather_features.shape}")
print(f"  ✓ Features per city: {weather_features.shape[1] - 3}")


[2/6] Loading Phase 2 weather features...
  ✓ Weather features: (39, 30)
  ✓ Features per city: 27


## Step 3: Load Embeddings (Phase 3)

In [13]:
print("\n[3/6] Loading Phase 3 embeddings...")

dest_embeddings = joblib.load(CACHE_DIR / 'destination_embeddings.joblib')
user_embeddings = joblib.load(CACHE_DIR / 'user_embeddings.joblib')

print(f"  ✓ Destination embeddings: {len(dest_embeddings)} cities")
print(f"  ✓ User embeddings: {len(user_embeddings)} users")
print(f"  ✓ Embedding dimension: 384")


[3/6] Loading Phase 3 embeddings...
  ✓ Destination embeddings: 39 cities
  ✓ User embeddings: 23 users
  ✓ Embedding dimension: 384


## Step 4: Build User and Destination Feature DataFrames

In [14]:
print("\n[4/6] Building feature dataframes...")

# Merge train_reviews with businesses to get city info
train_reviews_with_city = train_reviews.merge(
    businesses[['business_id', 'city']],
    on='business_id',
    how='left'
)

# Create user features from training data
user_features_list = []
for user_id in train_reviews_with_city['user_id'].unique():
    user_reviews = train_reviews_with_city[train_reviews_with_city['user_id'] == user_id]
    
    user_feat = {
        'user_id': user_id,
        'avg_rating': user_reviews['stars'].mean(),
        'rating_std': user_reviews['stars'].std(),
        'num_reviews': len(user_reviews),
        'num_cities': user_reviews['city'].nunique()
    }
    user_features_list.append(user_feat)

user_features_df = pd.DataFrame(user_features_list)
print(f"  ✓ User features: {user_features_df.shape}")

# Create destination features
dest_features_list = []
for city in businesses['city'].unique():
    city_businesses = businesses[businesses['city'] == city]
    city_reviews = train_reviews_with_city[train_reviews_with_city['city'] == city]
    
    dest_feat = {
        'city': city,
        'avg_rating': city_reviews['stars'].mean() if len(city_reviews) > 0 else 0,
        'num_reviews': len(city_reviews),
        'num_businesses': len(city_businesses),
    }
    dest_features_list.append(dest_feat)

dest_features_df = pd.DataFrame(dest_features_list)

# Merge with weather features
dest_features_df = dest_features_df.merge(
    weather_features,
    left_on='city',
    right_on='city',
    how='left'
)

print(f"  ✓ Destination features: {dest_features_df.shape}")


[4/6] Building feature dataframes...
  ✓ User features: (18, 5)
  ✓ Destination features: (39, 33)


## Step 5: Train Random Forest Model

In [15]:
print("\n[5/6] Training Random Forest model...")

# Initialize recommender
rf_recommender = RandomForestRecommender(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RF_RANDOM_STATE
)

# Build feature matrix for training
X_train, y_train = rf_recommender.build_feature_matrix(
    train_reviews_with_city,
    user_features_df,
    dest_features_df,
    dest_embeddings,
    user_embeddings
)

print(f"  ✓ Training set: {X_train.shape}")
print(f"  ✓ Features: {X_train.shape[1]}")
print(f"  ✓ Samples: {X_train.shape[0]}")

# Train model
rf_recommender.train(X_train, y_train)
print(f"  ✓ Model trained with {RF_N_ESTIMATORS} trees, max_depth={RF_MAX_DEPTH}")


[5/6] Training Random Forest model...
  ✓ Training set: (221, 804)
  ✓ Features: 804
  ✓ Samples: 221
  ✓ Model trained with 100 trees, max_depth=15


## Step 6: Evaluate on Test Set

In [16]:
print("\n[6/6] Evaluating on test set...")

# Merge test_reviews with businesses to get city info
test_reviews_with_city = test_reviews.merge(
    businesses[['business_id', 'city']],
    on='business_id',
    how='left'
)

# Build feature matrix for test
X_test, y_test = rf_recommender.build_feature_matrix(
    test_reviews_with_city,
    user_features_df,
    dest_features_df,
    dest_embeddings,
    user_embeddings
)

print(f"  ✓ Test set: {X_test.shape}")

# Predict
y_pred = rf_recommender.predict(X_test)

# Compute evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n{'='*70}")
print("RANDOM FOREST RESULTS")
print(f"{'='*70}")
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score:                  {r2:.4f}")
print(f"{'='*70}")


[6/6] Evaluating on test set...
  ✓ Test set: (77, 804)

RANDOM FOREST RESULTS
Mean Squared Error (MSE):  1.1461
Mean Absolute Error (MAE): 0.8836
R² Score:                  -0.2091


## Feature Importance Analysis

In [17]:
print("WEATHER FEATURE VERIFICATION")
print("="*70)

# Check weather_features dataframe
print(f"\nWeather features shape: {weather_features.shape}")
print(f"Weather features columns: {weather_features.columns.tolist()}")

# Check dest_features_df (should include weather now)
print(f"\nDestination features shape: {dest_features_df.shape}")
dest_weather_cols = [col for col in dest_features_df.columns if col not in ['city', 'avg_rating', 'num_reviews', 'num_businesses']]
print(f"Weather-related columns in dest_features_df: {dest_weather_cols[:10]}")

# Check if weather made it into the model features
print(f"\nTotal model features: {len(rf_recommender.feature_names)}")
weather_in_model = [f for f in rf_recommender.feature_names if 'dest_' in f and 'emb' not in f and 'rating' not in f and 'reviews' not in f and 'businesses' not in f]
print(f"Weather features in model: {len(weather_in_model)}")
if weather_in_model:
    print(f"  Examples: {weather_in_model[:5]}")
else:
    print("  ⚠️  NO WEATHER FEATURES FOUND!")

WEATHER FEATURE VERIFICATION

Weather features shape: (39, 30)
Weather features columns: ['city', 'lat', 'lon', 'temp_month_1', 'temp_month_2', 'temp_month_3', 'temp_month_4', 'temp_month_5', 'temp_month_6', 'temp_month_7', 'temp_month_8', 'temp_month_9', 'temp_month_10', 'temp_month_11', 'temp_month_12', 'precip_month_1', 'precip_month_2', 'precip_month_3', 'precip_month_4', 'precip_month_5', 'precip_month_6', 'precip_month_7', 'precip_month_8', 'precip_month_9', 'precip_month_10', 'precip_month_11', 'precip_month_12', 'annual_avg_temp', 'annual_precipitation', 'temp_range']

Destination features shape: (39, 33)
Weather-related columns in dest_features_df: ['lat', 'lon', 'temp_month_1', 'temp_month_2', 'temp_month_3', 'temp_month_4', 'temp_month_5', 'temp_month_6', 'temp_month_7', 'temp_month_8']

Total model features: 804
Weather features in model: 29
  Examples: ['dest_annual_avg_temp', 'dest_annual_precipitation', 'dest_lat', 'dest_lon', 'dest_precip_month_1']


In [18]:
print("\nTop 20 Most Important Features:")
print("-" * 50)

top_features = rf_recommender.get_feature_importance(top_n=20)
if top_features is not None:
    for idx, row in top_features.iterrows():
        print(f"{row['feature']:30s} {row['importance']:.4f}")
else:
    print("Feature importance not available")


Top 20 Most Important Features:
--------------------------------------------------
user_avg_rating                0.2408
dest_avg_rating                0.0822
user_emb_26                    0.0219
user_emb_298                   0.0113
user_emb_130                   0.0086
user_emb_21                    0.0076
user_emb_209                   0.0068
dest_emb_176                   0.0053
dest_emb_318                   0.0052
dest_emb_66                    0.0052
dest_emb_179                   0.0048
user_emb_336                   0.0048
dest_emb_302                   0.0047
user_emb_206                   0.0046
user_emb_263                   0.0046
user_emb_275                   0.0044
user_emb_242                   0.0044
user_emb_361                   0.0044
user_emb_286                   0.0043
dest_emb_244                   0.0038


## Save Model for Phase 5

In [19]:
print("\nSaving trained model...")

model_path = OUTPUT_DIR / 'random_forest_model.joblib'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save model and preprocessors
joblib.dump(rf_recommender, model_path)
print(f"  ✓ Model saved to {model_path}")

# Save results
results = {
    'mse': float(mse),
    'mae': float(mae),
    'r2': float(r2),
    'n_estimators': RF_N_ESTIMATORS,
    'max_depth': RF_MAX_DEPTH,
    'n_features': X_train.shape[1],
    'n_train_samples': X_train.shape[0],
    'n_test_samples': X_test.shape[0]
}

results_path = REPORT_DIR / 'phase4a_random_forest_results.txt'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(results_path, 'w') as f:
    f.write("PHASE 4a: Random Forest Results\n")
    f.write("="*50 + "\n")
    for k, v in results.items():
        f.write(f"{k:20s}: {v}\n")

print(f"  ✓ Results saved to {results_path}")


Saving trained model...
  ✓ Model saved to C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\outputs\random_forest_model.joblib
  ✓ Results saved to C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\reports\phase4a_random_forest_results.txt


## Summary

✅ **Phase 4a Complete: Random Forest Model**

**Model Configuration:**
- Estimators: 100
- Max depth: 15
- Features: ~800 dimensions (user + destination + embeddings + weather)

**Performance:**
- MSE: [calculated above]
- MAE: [calculated above]
- R²: [calculated above]

**Next Steps:**
- Phase 4b: Train Two-Tower Neural Network
- Phase 5: Run ablation experiments
- Phase 6: Final analysis

In [20]:
print("\n" + "="*70)
print("ABLATION TEST: MODEL PERFORMANCE WITHOUT WEATHER")
print("="*70)

# Identify weather feature indices
weather_feature_names = [f for f in rf_recommender.feature_names if 'dest_' in f and 'emb' not in f 
                         and 'rating' not in f and 'reviews' not in f and 'businesses' not in f]
weather_indices = [rf_recommender.feature_names.index(f) for f in weather_feature_names]

print(f"\nRemoving {len(weather_feature_names)} weather features...")

# Remove weather from train and test sets
X_train_no_weather = np.delete(X_train, weather_indices, axis=1)
X_test_no_weather = np.delete(X_test, weather_indices, axis=1)

# Train new model without weather
rf_no_weather = RandomForestRecommender(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RF_RANDOM_STATE
)
rf_no_weather.train(X_train_no_weather, y_train)

# Evaluate
y_pred_no_weather = rf_no_weather.predict(X_test_no_weather)
mse_no_weather = mean_squared_error(y_test, y_pred_no_weather)
mae_no_weather = mean_absolute_error(y_test, y_pred_no_weather)
r2_no_weather = r2_score(y_test, y_pred_no_weather)

# Compare
print(f"\n{'Metric':<25} {'With Weather':<20} {'Without Weather':<20} {'Difference':<15}")
print("-" * 80)
print(f"{'MSE':<25} {mse:<20.4f} {mse_no_weather:<20.4f} {mse - mse_no_weather:<+15.4f}")
print(f"{'MAE':<25} {mae:<20.4f} {mae_no_weather:<20.4f} {mae - mae_no_weather:<+15.4f}")
print(f"{'R² Score':<25} {r2:<20.4f} {r2_no_weather:<20.4f} {r2 - r2_no_weather:<+15.4f}")

# Conclusion
print(f"\n{'='*70}")
if abs(r2 - r2_no_weather) < 0.01:
    print(f"✓ CONCLUSION: Weather has minimal impact (ΔR² = {r2 - r2_no_weather:+.4f})")
    print("  → You can safely remove weather to simplify the model")
else:
    print(f"✓ CONCLUSION: Weather improves R² by {r2 - r2_no_weather:+.4f}")
    print("  → Keep weather features for better performance")
print(f"{'='*70}")


ABLATION TEST: MODEL PERFORMANCE WITHOUT WEATHER

Removing 29 weather features...

Metric                    With Weather         Without Weather      Difference     
--------------------------------------------------------------------------------
MSE                       1.1461               1.1797               -0.0336        
MAE                       0.8836               0.9045               -0.0208        
R² Score                  -0.2091              -0.2446              +0.0355        

✓ CONCLUSION: Weather improves R² by +0.0355
  → Keep weather features for better performance
